In [ ]:
import dsautils.calstatus as cs
from dsautils.dsa_store import DsaStore
from astropy.time import Time
import time
import datetime
import yaml
from dsacalib.weights import average_beamformer_solutions
import glob
import os
import numpy as np
from pkg_resources import resource_filename
import astropy.units as u
from dsautils import cnf
from dsacalib.plotting import summary_plot, plot_current_beamformer_solutions
from dsacalib.plotting import plot_beamformer_weights
from dsacalib.routines import get_files_for_cal, calibrate_measurement_set
from dsacalib.weights import get_good_solution, write_beamformer_solutions
from dsacalib.ms_io import convert_calibrator_pass_to_ms, uvh5_to_ms
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.backends.backend_pdf import PdfPages
import h5py
myconf = cnf.Conf(use_etcd=True)
ETCD = DsaStore()

# Calibration resources that may be useful

## Monitoring Calibration

The current calibration monitor points have not been integrated into influxdb and grafana.  You can check on the current status of calibration by pulling the relevant monitor points from etcd, shown below.

In [ ]:
# Check the rsync queue.  This task rsyncs correlator output from the
# correlator machines to dsa-storage.
# All tasks should be alive (i.e. ntasks_alive == ntasks_total), and
# queue_size should not exceed 16 (the number of files for a single time,
# one generated on each correlator node).
print('rsync:', ETCD.get_dict('/mon/cal/rsync_process'))
print('')
# Check the gathering task.  This task gathers the 16 files from the correlator
# machines for a single time. All tasks should be alive (i.e. ntasks_alive ==
# ntasks_total), and queue_size should not exceed 16. 
print('gather:', ETCD.get_dict('/mon/cal/gather_process'))
print('')
# Check the assessment queue.  This tasks checks to see if a calibrator exists
# in a given file.  It will only trigger calibration if the file contains the
# end of a 15-minute transit of the calibrator. All tasks should be alive
# (i.e. ntasks_alive == ntasks_total), and queue_size should not exceed a few.
print('assess:', ETCD.get_dict('/mon/cal/assess_process'))
print('')
# Check the status of the most recent calibration. A status of -1 means that
# calibration is incomplete (perhaps in progress).  A non-negative status means
# that calibration is complete.  After calibration, plots are generated and 
# the calibration table in Grafana is updated.  The status can be decoded with
# the dsautils.calstatus.decode function.
print('most recent calibration:', ETCD.get_dict('/mon/cal/calibration'))
status = ETCD.get_dict('/mon/cal/calibration')['status']
if status > -1:
    print(cs.decode(status))

## Averaging beamformer solutions

To plot the beamformer solutions and choose ones to average:

In [ ]:
CORR_PARAMS = myconf.get('corr')
CAL_PARAMS = myconf.get('cal')
MFS_PARAMS = myconf.get('fringe')

REFANTS = CAL_PARAMS['refant']
if isinstance(REFANTS, (str, int)):
    REFANTS = [REFANTS]
MSDIR = CAL_PARAMS['msdir']

BEAMFORMER_DIR = CAL_PARAMS['beamformer_dir']
print(BEAMFORMER_DIR)

ANTENNAS = np.array(list(CORR_PARAMS['antenna_order'].values()))
POLS = CORR_PARAMS['pols_voltage']
ANTENNAS_NOT_IN_BF = CAL_PARAMS['antennas_not_in_bf']
CORR_LIST = list(CORR_PARAMS['ch0'].keys())
#CORR_LIST = [int(cl.strip('lxd110h')) for cl in CORR_LIST]

In [ ]:
len(ANTENNAS)

In [ ]:
bfnames = [
    '0137+331_2022-09-13T10:02:36'
]
_ = plot_beamformer_weights(
    bfnames,
    ANTENNAS,
    BEAMFORMER_DIR,
    show=True
)

Making the averaged solution:

In [ ]:
with open(
    '{0}/beamformer_weights_{1}.yaml'.format(
        BEAMFORMER_DIR,
        bfnames[0]
    )
) as f:
    latest_solns = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
now = Time(datetime.datetime.utcnow())
now.precision = 0
averaged_files, avg_flags = average_beamformer_solutions(
    bfnames,
    now,
    BEAMFORMER_DIR,
    ANTENNAS,
    58849.0
)

In [ ]:
latest_solns['cal_solutions']['weight_files'] = averaged_files
latest_solns['cal_solutions']['source'] = [
    bfnames[0].split('_')[0]
]
latest_solns['cal_solutions']['caltime'] = [
    float(Time(bfnames[0].split('_')[1]).mjd)
]
for key, value in \
    latest_solns['cal_solutions']['flagged_antennas'].items():
    if 'casa solutions flagged' in value:
        value = value.remove('casa solutions flagged')
# Flag new bad solutions
idxant, idxpol = np.nonzero(avg_flags)
for i, ant in enumerate(idxant):
    key = '{0} {1}'.format(ANTENNAS[ant], POLS[idxpol[i]])
    if key not in \
        latest_solns['cal_solutions']['flagged_antennas'].keys():
        latest_solns['cal_solutions']['flagged_antennas'][key] = []
    latest_solns['cal_solutions']['flagged_antennas'][key] += \
        ['casa solutions flagged']
latest_solns['cal_solutions']['flagged_antennas'] = {
    key: value for key, value in
    latest_solns['cal_solutions']['flagged_antennas'].items()
    if len(value) > 0
}

In [ ]:
with open(
    '{0}/beamformer_weights_{1}.yaml'.format(
        BEAMFORMER_DIR, now.isot
    ),
    'w'
) as file:
    print('writing bf weights')
    _ = yaml.dump(latest_solns, file)

In [ ]:
with open(
    '{0}/beamformer_weights_{1}.yaml'.format(
        BEAMFORMER_DIR,now.isot
    )
) as f:
    latest_solutions = yaml.load(f, Loader=yaml.FullLoader)
ETCD.put_dict(
    '/mon/cal/bfweights',
    {
        'cmd': 'update_weights',
        'val': latest_solns['cal_solutions']
    }
)

## Manual Calibration - if calibration pipeline fails

In [ ]:
# The calibrator pass you want
#1156+314_2024-11-10T16:29:25
date = '2026-07-25'
calname = '1459+716'
dec = '+071p6'
# Parameters that don't need to be changed
antennas = [v for k, v in myconf.get('corr')['antenna_order'].items()]
duration = 15*u.min 
calsources = resource_filename(
    'dsacalib',
    f'data/calibrator_sources_dec{dec}.csv'
)
refcorr = 'corr03'
filelength = 5*u.min
msdir = '/operations/calibration/'
hdf5dir = '/operations/correlator/'
date_specifier = '{0}*'.format(date)
msname = '{0}/{1}_{2}'.format(msdir, date, calname)
print(calsources)

In [ ]:
# Get a list of the files for each calibrator
filenames = get_files_for_cal(
    calsources,
    hdf5dir,
    'sb01',
    duration,
    filelength,
    date_specifier
)
print(filenames)

In [ ]:
files = sorted(glob.glob(
    '/operations/correlator/{0}[{1}{2}{3}]???_sb??.hdf5'.format(
        filenames[date][calname]['files'][-1][:-4],
        int(filenames[date][calname]['files'][-1][-4])-3,
        filenames[date][calname]['files'][-1][-4],
        int(filenames[date][calname]['files'][-1][-4])+3
    )
), key = lambda x: x[-7:-5])


#files = sorted(glob.glob(
#    '/operations/correlator/{0}????_sb??.hdf5'.format(
#        ff[-1][:-4]
#    )
#), key = lambda x: x[-7:-5])

print(len(files),files)
#assert len(files) < 17
#print(filenames[date][calname]['files'][-1][:-4],int(filenames[date][calname]['files'][-1][-4])-3)

In [ ]:
ETCD.put_dict(
    '/cmd/cal',
    {
        'cmd': 'calibrate',
        'val':
        {
            'calname': calname,
            'flist': files
        }
    }
)

## Calibration within notebook

In [ ]:
import dsacalib.config as configuration
config = configuration.Configuration()
print(config)

In [ ]:
# manually make ms
cal = filenames[date][calname]['cal']
convert_calibrator_pass_to_ms(cal,date,filenames[date][calname]['files'],msdir=msdir,hdf5dir=hdf5dir,refmjd=config.refmjd)
#convert_calibrator_pass_to_ms(cal,'2025-06-12',['2025-06-12T00:48:26'],msdir=msdir,hdf5dir=hdf5dir,refmjd=config.refmjd)

In [ ]:
#uvh5_to_ms(files,"/operations/calibration/tmp",58849.0,ra=filenames[date][calname]['cal'].ra,dec=filenames[date][calname]['cal'].dec,flux=filenames[date][calname]['cal'].flux)
from dsacalib.uvh5_to_ms import load_uvh5_file
uvdata, pt_dec, ra, dec = load_uvh5_file("/operations/correlator/2024-06-12T23:30:18_sb12.hdf5", None, None, filenames[date][calname]['cal'].ra,filenames[date][calname]['cal'].dec)

In [ ]:
msname = '/operations/calibration/2024-06-18_0137+331'
status = calibrate_measurement_set(
    msname,
    filenames[date][calname]['cal'],
    refants=['103'],
    bad_antennas=['10'],
    bad_uvrange='2~27m',
)

In [ ]:
def extract_applied_delays(file):
    """Extracts the current snap delays from the hdf5 file.

    If delays are not set in the hdf5 file, uses the most recent delays in
    the beamformer weights directory instead.

    Parameters
    ----------
    file : str
        The full path to the hdf5 file.

    Returns
    -------
    ndarray
        The applied delays in ns.
    """
    with h5py.File(file, 'r') as f:
        if 'applied_delays_ns' in f['Header']['extra_keywords'].keys():
            delaystring = (
                f['Header']['extra_keywords']['applied_delays_ns']
                [()]
            ).astype(np.str)
            applied_delays = np.array(
                delaystring.split(' ')
            ).astype(np.int).reshape(-1, 2)
            applied_delays = applied_delays[ANTENNAS-1, :]
        else:
            current_solns = '{0}/beamformer_weights.yaml'.format(config.beamformer_dir)
            with open(current_solns) as yamlfile:
                calibration_params = yaml.load(
                    yamlfile,
                    Loader=yaml.FullLoader
                )['cal_solutions']
            applied_delays = np.array(calibration_params['delays'])*2
    return applied_delays

applied_delays = extract_applied_delays(files[0])
print(applied_delays)


In [ ]:
# correct order of delays

with open("beamformer_weights.yaml.bak3") as f:
    ow = yaml.load(f,yaml.Loader)
f.close()
correct_ao = ow['cal_solutions']['antenna_order']
correct_ao[27] = 101
correct_ao[28] = 9
correct_ao[29] = 8
for i in (list(np.arange(68,100))):
    correct_ao.append(i)
ao = list(ANTENNAS)

correct_delays = []
for i in ao:
    
    oi = 0
    for j,v in enumerate(correct_ao):
        if v==i:
            oi=j
    print(oi,i,applied_delays[oi])
    correct_delays.append(applied_delays[oi])

    

In [ ]:
#ttime = filenames[date][calname]['transit_time']
#ttime.precision = 0
msname = '/operations/calibration/2024-11-10_1156+314'
calname = '1156+314'
# Write beamformer solutions for one source
_ = write_beamformer_solutions(
    msname,
    calname,
    ttime,
    ANTENNAS,
    correct_delays,
    config.beamformer_dir,
    config.pols,
    config.nchan,
    config.nchan_spw,
    config.bw_GHz,
    config.chan_ascending,
    config.f0_GHz,
    config.ch0,
    config.refmjd,
    flagged_antennas=ANTENNAS_NOT_IN_BF
)

In [ ]:
# plot bf weights
import struct
sbs = ['00','01','02','03','04','05','06','07','08','09','10','11','12','13','14','15']
fls = []
for i in np.arange(16):
    fls.append("/operations/beamformer_weights/generated//beamformer_weights_sb"+sbs[i]+"_1156+314_2024-11-16T09:53:50.dat")

full_weights = np.zeros((96,16,48,2,2),dtype = np.float32)
ant_loc = []
for i in np.arange(16):
    f = open(fls[i],'rb')
    data = f.read()
    vv = np.asarray(struct.unpack("<18624f",data)).astype(np.float32)
    vals = vv[2*96:]
    ant_loc.append(vv[:2*96]) 
    vals = vals.reshape((96,48,2,2))
    full_weights[:,i,:,:,:] = vals
    f.close()
w = np.zeros((96,16,48,2),dtype = np.complex64)
w = 1.*full_weights[:,:,:,:,0] + 1j*full_weights[:,:,:,:,1]
w = w.reshape((96,768,2))

fls = []
for i in np.arange(16):
    fls.append("/home/ubuntu/dsa-notebooks/tmp/beamformer_weights_sb"+sbs[i]+"_1156+314_2024-11-10T16:29:25.dat")

full_weights = np.zeros((96,16,48,2,2),dtype = np.float32)
ant_loc = []
for i in np.arange(16):
    f = open(fls[i],'rb')
    data = f.read()
    vv = np.asarray(struct.unpack("<18624f",data)).astype(np.float32)
    vals = vv[2*96:]
    ant_loc.append(vv[:2*96]) 
    vals = vals.reshape((96,48,2,2))
    full_weights[:,i,:,:,:] = vals
    f.close()
wn = np.zeros((96,16,48,2),dtype = np.complex64)
wn = 1.*full_weights[:,:,:,:,0] + 1j*full_weights[:,:,:,:,1]
wn = w.reshape((96,768,2))

In [ ]:
plt.figure(figsize=(15,30))

for i in np.arange(12):
    
    plt.subplot(6,2,i+1)
    for j in np.arange(8):
        
        ant = i*8+j
        plt.plot(np.abs(w[ant,:,1])-np.abs(wn[ant,:,1]),label=str(config.antennas[ant]))
        #plt.plot(np.angle(wn[ant,:,1]),label=str(config.antennas[ant]))
        plt.xlim(0,100)
        
    plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(12,12))

x = ant_loc[0][:96]
y = ant_loc[0][96:]
plt.plot(x,y,'ko')